# CME538 — Introduction to Data Science
## Tutorial 8 — Classification, Evaluation & Cross-Validation

### Prediction of Rain Tomorrow

In this tutorial, we will build a machine learning model to predict whether it will rain tomorrow using weather observations from Australia.

Because the target has two possible outcomes  **Yes** or **No** this is a **binary classification** problem.

By the end of this tutorial, we will be able to:

1. Understand classification as a prediction problem.
2. Fit a logistic regression classifier.
3. Interpret predicted probabilities.
4. Convert probabilities into class predictions using a threshold.
5. Evaluate a classifier using accuracy, precision, recall, F1-score, and a confusion matrix.
6. Visualize classifier performance using ROC and precision-recall curves.
7. Understand decision boundaries.
8. Use cross-validation to estimate model performance.
9. Tune logistic regression hyperparameters.
10. Understand why class imbalance can make accuracy misleading.

### Tutorial structure

1. [Setup](#setup)
2. [Explore the data](#explore)
3. [Prepare the data](#prepare)
4. [Train a logistic regression classifier](#training)
5. [Probabilities and thresholding](#probabilities)
6. [Evaluate the classifier](#evaluation)
7. [Visual evaluation](#visual)
8. [Cross-validation and hyperparameter tuning](#cv)
9. [Class imbalance](#imbalance)
10. [Summary](#summary)

## 1. Classification problem

Suppose our client wants to know whether it will rain tomorrow.

We can represent this as:

$$
X \rightarrow y
$$

where:

- $X$ represents the weather measurements available today.
- $y$ represents whether it rains tomorrow.

Our target variable is `RainTomorrow`.

This is a **classification** problem because the target represents a category rather than a continuous numerical value.

In this tutorial, there are two classes:

$$
y \in \{\text{No}, \text{Yes}\}
$$

This makes the problem a **binary classification** problem.

### Classification versus regression

In regression, we predict a numerical quantity such as:

> What will the car price be?

In classification, we predict a category such as:

> Will it rain tomorrow?

<a id="setup"></a>

## 1. Setup

We begin by importing the packages required for data manipulation, visualization, preprocessing, model fitting, and evaluation.

This tutorial uses a modern scikit-learn workflow based on `Pipeline` and `ColumnTransformer`.

This is preferable to manually transforming the training and test sets because it helps ensure that preprocessing is learned from the training data only.

In [ ]:
# NumPy provides numerical operations and array functionality.
# Pandas is used for loading and manipulating tabular data.
# Matplotlib and Seaborn are used for visualization.
# Used to split the dataset into training and test sets.
# ColumnTransformer allows us to apply different preprocessing
# steps to numerical and categorical variables.
# Pipeline keeps preprocessing and model fitting together.
# Impute missing values using statistics learned from the training data.
# Standardize numerical features.
# Logistic regression is our classification model.
# Classification metrics.
# Cross-validation and hyperparameter tuning.
# Reproducibility.
# Make plots easier to read.


<a id="explore"></a>

## 2. Explore the data

We use the `weatherAUS.csv` dataset.

Each row represents a weather observation at a particular location and date.

Our target variable is:

- `RainTomorrow` — whether it rained the following day.

The remaining columns contain weather measurements that may help predict this outcome.

In [ ]:
# Load the weather dataset.
# Display the first five observations.


### Inspect the dataset

Before fitting a model, we should understand:

- how many observations we have,
- which variables are available,
- which variables are numerical,
- which variables are categorical,
- and where missing values occur.

In [ ]:
# Display the number of rows and columns.
# Display column names, data types, and non-null counts.
# Count missing values in each column.


### The target variable

Our target is `RainTomorrow`.

Since the target has two classes, this is a **binary classification** problem.

We first remove observations where the target itself is missing. A model cannot learn whether tomorrow will rain if the answer is unknown.

In [ ]:
# Remove observations where the target value is missing.
# Examine the available target classes.
# Show the proportion of each class.


### Class distribution

It is useful to visualize the target distribution.

This will also introduce an important issue later in the tutorial: **class imbalance**.

If one class is much more common than the other, a model can achieve high accuracy simply by predicting the majority class most of the time.

In [ ]:
# Plot the number of observations in each target class.


## 3. Prepare the data

Machine learning models require numerical inputs.

Our dataset contains both:

- **numerical variables**, such as temperature and pressure;
- **categorical variables**, such as location and wind direction.

We will also extract useful information from the date.

Rather than manually transforming the training and test sets, we will use a scikit-learn preprocessing pipeline.

In [ ]:
# Convert Date from text into a datetime object.
# Extract useful components of the date.
# The original Date column is no longer needed.
# Separate predictors from the target.
# Display the target and feature dimensions.


### Train-test split

We now separate the data into:

- **training data** — used to learn the model;
- **test data** — held back until the end to estimate performance on unseen observations.

We use `stratify=y` so that the proportion of Rain/No Rain observations is approximately preserved in both sets.

In [ ]:
# Split the data into training and test sets.
# Display the resulting sizes.
# Compare class proportions in the two sets.


### Identify numerical and categorical features

We will let the preprocessing pipeline handle:

- missing numerical values using the median;
- missing categorical values using the most frequent category;
- scaling of numerical variables;
- one-hot encoding of categorical variables.

This avoids manually modifying the test set using information calculated from the test data.

In [ ]:
# Identify numerical columns.
# Identify categorical columns.


### Build the preprocessing pipeline

We create two preprocessing pipelines.

For numerical variables:

1. Replace missing values with the training-set median.
2. Standardize the variables.

For categorical variables:

1. Replace missing values with the most frequent category.
2. Convert categories into one-hot encoded variables.

The `ColumnTransformer` applies the appropriate pipeline to each type of feature.

In [ ]:
# Numerical preprocessing:
# - median imputation handles missing numerical values;
# - StandardScaler puts numerical variables on a comparable scale.
# Categorical preprocessing:
# - most_frequent imputation handles missing categories;
# - OneHotEncoder converts categories into 0/1 indicators.
# Apply the appropriate preprocessing to each feature type.


<a id="training"></a>

## 4. Train a Logistic Regression Classifier

Logistic regression is a classification model that estimates the probability of belonging to a class.

For binary classification, the model estimates:

$$
P(Y=1\mid X)
$$

The logistic function converts the model's linear score into a probability between 0 and 1.

Unlike ordinary linear regression, logistic regression is designed for a categorical target.

In [ ]:
# Create the complete machine learning pipeline.
# Preprocessing happens first, followed by logistic regression.
# Fit the entire pipeline using the training data.


## 5. Classification versus regression

The key difference is the type of prediction.

### Regression

Predicts a numerical quantity:

$$
\hat{y} = 52,000
$$

For example, predicting a car price.

### Classification

Predicts a class:

$$
\hat{y} = \text{Yes}
$$

For example, predicting whether it will rain tomorrow.

However, a classifier can also provide a **probability** rather than only a class label.

<a id="probabilities"></a>

## 6. Predicted probabilities and thresholding

A classifier does not fundamentally have to say only "Yes" or "No".

For each observation, logistic regression can estimate:

$$
P(\text{RainTomorrow}=\text{Yes}\mid X)
$$

For example:

- probability = 0.10 → low chance of rain
- probability = 0.55 → moderate chance of rain
- probability = 0.90 → high chance of rain

To convert probabilities into class predictions, we use a **threshold**.

The default threshold is usually 0.50.

In [ ]:
# Predict the probability of each class on the test set.
# Display the first five probability predictions.


In [ ]:
# Identify which probability column corresponds to "Yes".
# Find the column corresponding to RainTomorrow = Yes.
# Extract the probability of rain.
# Display the first ten probabilities.


### Changing the classification threshold

With a threshold of 0.50:

$$
P(\text{Rain}) \geq 0.50
\Rightarrow \text{predict Yes}
$$

Otherwise, we predict No.

But 0.50 is not always the best threshold.

For example, if missing a rainy day is especially costly, we may prefer a lower threshold so that the model predicts "Yes" more often.

In [ ]:
# Convert probabilities into predictions using the default 0.50 threshold.
# Display the first ten predictions.


<a id="evaluation"></a>

## 7. Evaluating a classifier

A classifier can make four types of predictions:

| | Actual No | Actual Yes |
|---|---:|---:|
| Predicted No | True Negative (TN) | False Negative (FN) |
| Predicted Yes | False Positive (FP) | True Positive (TP) |

These four values form the **confusion matrix**.

Different metrics emphasize different types of errors.

In [ ]:
# Calculate the confusion matrix.
# Display the matrix.


### Accuracy

Accuracy is the proportion of predictions that are correct:

$$
Accuracy =
\frac{TP+TN}{TP+TN+FP+FN}
$$

Accuracy is easy to understand, but it can be misleading when classes are imbalanced.

In [ ]:
# Calculate test-set accuracy.


### Precision, Recall and F1-score

For the positive class, "Yes":

**Precision**

Of the observations predicted as Yes, how many actually were Yes?

$$
Precision = \frac{TP}{TP+FP}
$$

**Recall**

Of the observations that actually were Yes, how many did we correctly identify?

$$
Recall = \frac{TP}{TP+FN}
$$

**F1-score**

The harmonic mean of precision and recall:

$$
F1 = 2\frac{Precision\cdot Recall}
{Precision+Recall}
$$

There is usually a trade-off between precision and recall.

In [ ]:
# Calculate precision, recall, and F1-score for the Rain = Yes class.


In [ ]:
# classification_report provides several useful metrics at once.


## 8. Confusion matrix visualization

A confusion matrix is often easier to interpret visually.

The rows represent the **actual** class and the columns represent the **predicted** class.

In [ ]:
# Create a heatmap of the confusion matrix.


<a id="visual"></a>

## 9. Visual metrics: ROC curve and AUC

The ROC curve evaluates the classifier over many possible probability thresholds.

It plots:

- **True Positive Rate (TPR)** against
- **False Positive Rate (FPR)**.

The area under the ROC curve is called **ROC-AUC**.

A value close to:

- 1 → strong discrimination;
- 0.5 → approximately random ranking.

In [ ]:
# Calculate the false-positive and true-positive rates
# across many possible probability thresholds.
# Calculate the area under the ROC curve.
# Plot the ROC curve.
# Random guessing corresponds approximately to this diagonal.


### Precision-recall curve

When the positive class is relatively uncommon, the precision-recall curve can be especially informative.

It shows the trade-off between:

- precision;
- recall;

as the classification threshold changes.

In [ ]:
# Calculate precision and recall at different thresholds.
# Calculate average precision.
# Plot the precision-recall curve.


## 10. Decision boundaries

A classifier separates observations into different classes.

For a linear classifier such as logistic regression, the decision boundary is determined by:

$$
\theta_0+\theta_1x_1+\cdots+\theta_px_p = 0
$$

For two features, this corresponds to a line.

One side of the boundary is classified as one class, while the other side is classified as the other class.

In this tutorial we have many features, so the boundary exists in a higher-dimensional feature space and cannot be directly drawn as a simple 2-D line.

<a id="cv"></a>

### 11. Cross-validation with classification metrics

We use **StratifiedKFold** so that each fold maintains approximately the
same proportion of the target classes.

Because our target variable contains the labels `"No"` and `"Yes"`, we
must explicitly tell the precision, recall, and F1 scorers that `"Yes"`
is the positive class.

This is important because these metrics depend on which class is treated
as the positive class.

In [ ]:
# StratifiedKFold preserves the class proportions in each fold.
# ---------------------------------------------------------
# Define the positive class explicitly.
#
# Our target contains "No" and "Yes", so we define "Yes"
# as the positive class for precision, recall, and F1.
# ---------------------------------------------------------
# ---------------------------------------------------------
# Evaluate the classification model using cross-validation.
#
# ROC AUC is calculated from the model's probability scores,
# while precision/recall/F1 use "Yes" as the positive class.
# ---------------------------------------------------------
# ---------------------------------------------------------
# Display the mean and standard deviation for each metric.
#
# Mean = average performance across the five folds.
# Standard deviation = how much performance varies between folds.
# ---------------------------------------------------------


## 12. Hyperparameter tuning

Logistic regression has hyperparameters that control model complexity.

One important hyperparameter is `C`.

`C` controls the strength of regularization:

- smaller `C` → stronger regularization;
- larger `C` → weaker regularization.

Instead of choosing `C` arbitrarily, we can evaluate several values using cross-validation.

In [ ]:
# Define candidate values of the regularization parameter C.
# GridSearchCV evaluates each candidate using cross-validation.
# Fit the grid search using training data only.
# Display the best hyperparameter and cross-validation score.


### Final test-set evaluation

The test set has not been used to choose the model or hyperparameters.

We can therefore use it once to obtain our final estimate of generalization performance.

In [ ]:
# Generate test-set probabilities using the best model.
# Convert probabilities into class predictions using a 0.50 threshold.
# Evaluate the final model on unseen test data.


<a id="imbalance"></a>

## 13. Class imbalance

Our target is not perfectly balanced: there are more "No" observations than "Yes" observations.

This matters because accuracy can hide poor performance on the minority class.

For example, imagine that:

- 90% of observations are No;
- 10% are Yes.

A classifier that always predicts No would have 90% accuracy while detecting **zero** rainy days.

Therefore, when classes are imbalanced, we should consider metrics such as:

- precision;
- recall;
- F1-score;
- ROC-AUC;
- precision-recall curves.

The appropriate metric depends on the cost of different errors.

In [ ]:
# Calculate the accuracy of a naive majority-class classifier.


### Class weighting

One way to make logistic regression pay more attention to the minority class is to use:

```text
class_weight="balanced"
```

In [ ]:
# Create a version of the model that automatically gives
# more weight to the minority class.
# Fit the balanced model using training data.
# Generate probabilities and predictions.
# Compare classification performance.


## 14. Threshold trade-off

The default threshold of 0.50 is only one possible choice.

A lower threshold generally makes the classifier predict "Yes" more often:

- recall tends to increase;
- precision may decrease.

A higher threshold generally makes the classifier more conservative:

- precision may increase;
- recall may decrease.

The appropriate threshold should therefore depend on the application and the relative cost of false positives and false negatives.

In [ ]:
# Examine how the threshold changes precision and recall.
    # Convert probabilities to class predictions.
    # Calculate metrics for this threshold.


In [ ]:
# Plot precision and recall as the classification threshold changes.
